In [ ]:
# Kaggle Notebook: LSTM Deep Learning Optimization (Sintetik 90)
# Petunjuk:
# 1. Pastikan Anda mengaktifkan GPU (P100 atau T4 x2) di Kaggle.
# 2. Upload file 'processed_tropical_features.csv'
# 3. Klik "Run All" dan tunggu keajaiban EarlyStopping bekerja.

import pandas as pd
import numpy as np
import os
import shutil
import joblib
import json
import time
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
from IPython.display import FileLink, display

# TensorFlow / Keras
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings('ignore')

# ==========================================
# KONFIGURASI
# ==========================================
DATA_FILE = "processed_tropical_features.csv" 
ARTIFACTS_DIR = "agrisense_lstm_optimized_models"
AUDIT_FILE = os.path.join(ARTIFACTS_DIR, "tahap5_lstm_audit.json")

os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TARGET_MAP = {
    'target_co2_ppm': 'CO2 (ppm)',
    'target_nee_agrisense': 'Carbon Flux (NEE AgriSense)',
    'target_carbon_potential_score': 'Carbon Potential Score',
    'target_kelembapan_tanah': 'Soil Moisture (%)',
    'target_ph_tanah': 'pH Tanah'
}

# Parameter Optimasi LSTM
SEQ_LEN = 24       
HORIZON = 24       
BATCH_SIZE = 256    # Diperbesar agar GPU lebih maksimal memprosesnya
EPOCHS = 500        # Iterasi Raksasa
INITIAL_LR = 0.002  # Sedikit lebih besar untuk awal

def create_sequences(data_features, data_targets, seq_len, horizon):
    xs, ys = [], []
    for i in range(len(data_features) - seq_len - horizon + 1):
        x = data_features[i : i + seq_len]
        y = data_targets[i + seq_len + horizon - 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

print("="*60)
print("1. MEMUAT DATASET (38 Fitur)")
print("="*60)
df = pd.read_csv(DATA_FILE)

available_targets = [t for t in TARGET_MAP.keys() if t in df.columns]
feature_columns = [c for c in df.columns if c not in TARGET_MAP.keys() and c != 'TIMESTAMP']

feature_df = df[feature_columns].copy()
feature_df.ffill(inplace=True)
feature_df.bfill(inplace=True)

# ==========================================
# PRE-PROCESSING UNTUK LSTM
# ==========================================
lstm_scaler = StandardScaler()
X_lstm = lstm_scaler.fit_transform(feature_df)
joblib.dump(lstm_scaler, os.path.join(ARTIFACTS_DIR, "lstm_feature_scaler.joblib"))
feature_dim = X_lstm.shape[1]

audit_results = {"LSTM_Optimized": {}}

print("\n" + "="*60)
print("2. MEMULAI DEEP LEARNING OPTIMIZATION (LSTM)")
print(f"Max Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | GPU Accelerated")
print("="*60)

for target in available_targets:
    target_label = TARGET_MAP[target]
    print(f"\n🚀---> OPTIMASI LSTM: {target_label} <---🚀")
    
    # Target Scaler
    target_scaler = StandardScaler()
    scaled_target = target_scaler.fit_transform(df[[target]])
    joblib.dump(target_scaler, os.path.join(ARTIFACTS_DIR, f"lstm_target_scaler_{target_label}.joblib"))
    
    # Sequences
    X_seq, y_seq = create_sequences(X_lstm, scaled_target.flatten(), SEQ_LEN, HORIZON)
    
    split_idx = int(len(X_seq) * 0.8)
    X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
    y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]
    
    # Arsitektur Jaringan (Lebih Cerdas)
    model = Sequential([
        LSTM(128, activation='tanh', return_sequences=True, input_shape=(SEQ_LEN, feature_dim)),
        Dropout(0.25),
        LSTM(64, activation='tanh'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=INITIAL_LR)
    model.compile(optimizer=optimizer, loss='mse')
    
    model_path = os.path.join(ARTIFACTS_DIR, f"lstm_model_{target_label}_h{HORIZON}.keras")
    
    # CALLBACKS SENSOR CERDAS
    # 1. EarlyStopping: Berhenti jika dalam 30 iterasi tidak ada perbaikan
    early_stopping = EarlyStopping(
        monitor='val_loss', patience=30, restore_best_weights=True, verbose=1
    )
    # 2. ReduceLROnPlateau: Turunkan kecepatan belajar jika mentok selama 10 iterasi
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=10, min_lr=0.00001, verbose=1
    )
    # 3. ModelCheckpoint: Selalu simpan versi ter-PINTAR di setiap iterasi
    checkpoint = ModelCheckpoint(
        model_path, monitor='val_loss', save_best_only=True, verbose=0
    )
    
    start_time = time.time()
    
    history = model.fit(
        X_train, y_train, 
        epochs=EPOCHS, 
        batch_size=BATCH_SIZE, 
        validation_split=0.15, # 15% dari data train untuk validasi early stopping
        callbacks=[early_stopping, reduce_lr, checkpoint],
        verbose=1
    )
    
    time_taken = (time.time() - start_time) / 60.0
    
    # Evaluate
    # Kita muat ulang model terbaik yang diselamatkan oleh checkpoint
    best_model = tf.keras.models.load_model(model_path, compile=False)
    pred = best_model.predict(X_test, verbose=0)
    
    pred_real = target_scaler.inverse_transform(pred)
    actual_real = target_scaler.inverse_transform(y_test.reshape(-1, 1))
    
    r2 = r2_score(actual_real, pred_real)
    mae = mean_absolute_error(actual_real, pred_real)
    
    best_epoch = len(history.history['loss'])
    
    print(f"[*] Total Waktu     : {time_taken:.2f} menit")
    print(f"[*] Berhenti di Epoch: {best_epoch} (EarlyStopping Activated)")
    print(f"[*] R2 Score Akhir  : {r2:.4f}")
    print(f"[*] MAE Akhir       : {mae:.4f}")
    
    audit_results["LSTM_Optimized"][target_label] = {
        "R2": float(r2),
        "MAE": float(mae),
        "Epochs_Ran": int(best_epoch),
        "Time_Minutes": float(time_taken)
    }

with open(AUDIT_FILE, 'w') as f:
    json.dump(audit_results, f, indent=4)

# ==========================================
# ZIPPING & DOWNLOAD
# ==========================================
print("\n" + "="*60)
print("OPTIMASI SELESAI! SEDANG MELAKUKAN ZIPPING...")

ZIP_NAME = "agrisense_lstm_optimized_models"
shutil.make_archive(ZIP_NAME, 'zip', ARTIFACTS_DIR)

print("="*60)
print("ZIP SELESAI! Silakan klik link di bawah ini untuk mengunduh:")
display(FileLink(f'{ZIP_NAME}.zip'))
print("="*60)
